[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Bienvenida, reglas del taller y entorno de trabajo.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 1: *Fundamentos de sistemas agénticos*

En este *notebook* se presentan las reglas del taller, se configuran las claves de API como secretos de *GitHub Codespaces* y se verifica que el modelo, *LangSmith* y el servidor MCP del taller respondan.

---
## 1. Bienvenida y reglas del taller.

* El taller consta de **8 bloques de 2 horas** y combina teoría, casos de diseño y laboratorios.
* Los *notebooks* `02` a `14` tienen **ejercicios**; las versiones resueltas están en `soluciones/`. El `15` es la demo final del instructor.
* A lo largo de los laboratorios se construye un mismo sistema: el **asistente de soporte interno** de *Andina Logística*, una empresa ficticia. Consulta políticas (`data/politicas/`), gestiona tickets mediante un servidor MCP (`mcp_servers/tickets_server.py`) y escala casos a una persona.
* **Modelo del taller: *Gemini*.** Todos los ejemplos, ejercicios y soluciones usan `google_genai:gemini-3.8-flash` y los *embeddings* `google_genai:gemini-embedding-001`. *LangChain* funciona con cualquier proveedor (*OpenAI*, *Anthropic*, modelos abiertos…) cambiando una sola cadena, y el *notebook* `03` lo demuestra; pero en clase **todos usaremos *Gemini*** para comparar resultados en igualdad de condiciones. Las claves de otros proveedores son opcionales.
* Los modelos de lenguaje no son deterministas: tus respuestas no serán idénticas a las del instructor ni a las de tus compañeros. Lo que importa es el **comportamiento**: qué herramientas llama el agente, en qué orden y con qué argumentos.
* Cada llamada consume *tokens* (fragmentos de texto; se definen en la sección 3) de **tu** clave de API. Las celdas están pensadas para ejecutarse pocas veces; evita bucles grandes.

| Módulo | Tema | *Notebooks* |
|---|---|---|
| 1 | Claves de API y fundamentos de sistemas agénticos | `00`-`02` |
| 2 | El ecosistema *LangChain* v1 | `03`-`04` |
| 3 | Herramientas, contexto y RAG | `05`-`06` |
| 4 | MCP y comunicación entre agentes | `07` |
| 5 | *Middleware*: controlar el ciclo del agente | `08` |
| 6 | *LangGraph*: estado, durabilidad y multiagente | `09`-`10` |
| 7 | Memoria y *Deep Agents* | `11`-`12` |
| 8 | Evaluación y observabilidad | `13` |
| 9 | Seguridad, costos y despliegue | `14` |
| 10 | Demo final integradora | `15` |

---
## 2. Claves de API como secretos de *Codespaces*.

Las claves de API se crearon y guardaron como secretos de *Codespaces* en el *notebook* [`00_claves_de_api_y_secretos.ipynb`](00_claves_de_api_y_secretos.ipynb). Ahí están los pasos, los errores comunes y qué hacer si una clave se filtra; aquí solo se comprueba que el *Codespace* las vea.

* La siguiente celda indicará qué claves están disponibles **sin mostrar su valor**. `GOOGLE_API_KEY` debe aparecer como configurada. Nunca imprimas el valor de una clave: la salida de la celda se guarda dentro del *notebook*.

In [ ]:
import os

{variable: ("✅ configurada" if os.environ.get(variable) else "❌ falta")
 for variable in ["GOOGLE_API_KEY", "LANGSMITH_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY"]}

---
## 3. Verificación del entorno.

* La siguiente celda mostrará la versión de *Python* y de las bibliotecas del taller. Debe ser *Python* 3.12 y *LangChain* 1.4.

In [ ]:
import sys
from importlib.metadata import version

paquetes = ["langchain", "langchain-core", "langgraph", "langsmith",
            "langchain-google-genai", "langchain-mcp-adapters", "deepagents"]
{"Python": sys.version.split()[0], **{p: version(p) for p in paquetes}}

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (solo en ejecución local), definirá el modelo del taller (*Gemini*) y verificará que exista la clave `GOOGLE_API_KEY`. Si falla, sigue el *notebook* `00_claves_de_api_y_secretos.ipynb`.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local; en Codespaces se usan los secretos

# Este taller usa Gemini. Cualquier proveedor funciona cambiando la cadena "proveedor:modelo" (ver notebook 03).
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.8-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if MODELO.startswith("google_genai") and not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Falta GOOGLE_API_KEY. Sigue el notebook 00_claves_de_api_y_secretos: guárdala como secreto de Codespaces "
                       "con acceso a este repositorio y vuelve a abrir el Codespace.")
MODELO

* La siguiente celda hará la primera llamada a *Gemini*. `init_chat_model` crea el modelo a partir de la cadena `"proveedor:modelo"`; el *notebook* `03` lo explica. Si responde `listo`, la clave funciona.

In [ ]:
from langchain.chat_models import init_chat_model

modelo = init_chat_model(MODELO)
respuesta = modelo.invoke("Responde solo con la palabra: listo")
respuesta.text

### Conceptos básicos del modelo.

| Concepto | Qué es |
|---|---|
| ***LLM*** (modelo de lenguaje grande) | Un modelo entrenado con grandes volúmenes de texto que, a partir de una entrada, genera la continuación más probable. *Gemini* es un *LLM*. |
| ***Token*** | El fragmento de texto con el que el modelo lee y escribe: una palabra, parte de una palabra o un signo. El uso, los límites y el costo se miden en *tokens*. |
| **Mensaje de sistema** | Las instrucciones que fijan el papel y las reglas del modelo durante toda la conversación. Se envía antes de los mensajes del usuario. |
| **Ventana de contexto** | El máximo de *tokens* que el modelo puede considerar en una llamada: instrucciones, historial, resultados de herramientas y su propia respuesta. Lo que no cabe, el modelo no lo ve. |

* La siguiente celda mostrará cuántos *tokens* de entrada y de salida usó la llamada anterior.

In [ ]:
respuesta.usage_metadata

* Si configuraste `LANGSMITH_API_KEY`, abre *smith.langchain.com* → proyecto **py501**: la llamada anterior debe aparecer como una traza. Una **traza** es el registro de los pasos de una ejecución: llamadas al modelo, herramientas usadas y sus resultados.

* **MCP** (*Model Context Protocol*) es un protocolo estándar para que un agente use las herramientas y los datos de otro programa, llamado **servidor MCP**. Se estudia en el *notebook* `07`.
* La siguiente celda arrancará el servidor MCP de tickets y listará sus herramientas. En *Jupyter* se usa `await` directamente porque el *notebook* ya tiene un *event loop* corriendo: **no** uses `asyncio.run()`.

In [ ]:
import sys

from langchain_mcp_adapters.client import MultiServerMCPClient

cliente = MultiServerMCPClient({"tickets": {
    "transport": "stdio", "command": sys.executable,
    "args": [os.path.join(RAIZ, "mcp_servers", "tickets_server.py")]}})
[h.name for h in await cliente.get_tools()]

---
## ✅ Cierre.

**Práctica:** verificar que las cuatro comprobaciones (claves, versiones, llamada a *Gemini* y servidor MCP) funcionen antes de continuar con el *notebook* `02`. Si alguna falla, consulta la sección 7 de `00_claves_de_api_y_secretos.ipynb`.

**Siguiente *notebook*:** `02_que_es_un_agente.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>